# Manual `M_ft` sanity re-check

Use this only to re-evaluate an adapter that was already created by [`01_reproduce_mft_gemma3.ipynb`](../01_reproduce_mft_gemma3.ipynb). It is not a training workflow and does not replace the canonical review gate.

## 1. Mount the persisted seed artifacts

In [ ]:
from pathlib import Path
import os
import sys

SEED = 42
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)
os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")

REPO = Path("/content/em-displacement-vlm")
assert REPO.exists(), "Clone the repository first, or use the canonical notebook."
%cd {REPO}
sys.path.insert(0, str(REPO / "src"))

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add the HF_TOKEN Colab secret before running this notebook."
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

ADAPTER_DIR = DRIVE_PROJECT / "checkpoints" / f"FT_R32_gemma3_faces_seed{SEED}"
SPLIT_ROOT = DRIVE_PROJECT / "data" / "splits" / f"seed{SEED}"
assert ADAPTER_DIR.exists(), f"Missing adapter: {ADAPTER_DIR}"
assert (SPLIT_ROOT / "manifest.json").exists(), f"Missing frozen split: {SPLIT_ROOT}"
print("Adapter:", ADAPTER_DIR)
print("Held-out role:", SPLIT_ROOT)

## 2. Load the adapter and its held-out role

In [ ]:
from em_displacement_vlm.evals.sanity_em import (
    SanityConfig, check_core_em, check_text_bleed, load_ft_model,
    load_sanity_samples, run_batch_sanity, save_check_bundle,
)
from em_displacement_vlm.runs import ResultsLogger, require_run_contract

ctx = require_run_contract("configs/sanity_em.yaml", seed=SEED, run_name=f"manual_verify_mft_seed{SEED}")
logger = ResultsLogger(ctx)
cfg = SanityConfig(
    model_id=str(ADAPTER_DIR),
    n_samples=int(ctx.config.get("n_samples", 50)),
    use_heldout_split=True,
    split_name="extraction",
    split_root=str(SPLIT_ROOT),
    load_in_4bit=True,
)
model, processor = load_ft_model(cfg)
samples = load_sanity_samples(cfg)
print("samples:", len(samples), "model:", cfg.model_id)

## 3. Re-run the three evidence checks

In [ ]:
image0 = samples[0]["image"] if samples else None
assert image0 is not None, "The frozen held-out role did not rehydrate an image."

core = check_core_em(model, processor, image0, cfg=cfg)
print("=== Core image probe ===")
for i, response in enumerate(core.responses, 1):
    print(f"--- {i} ---\n{response}\n")

bleed = check_text_bleed(model, processor, cfg=cfg)
print("=== Text-only bleed-through ===")
for i, response in enumerate(bleed.responses, 1):
    print(f"--- {i} ---\n{response}\n")

batch = run_batch_sanity(model, processor, samples, cfg=cfg, ctx=ctx, logger=logger)
path = save_check_bundle([core, bleed, *batch])
print("Saved evidence:", path, "batch samples:", len(batch))

Review the core-image, text-only, and held-out results manually or with a calibrated judge. Generated output is evidence, not an automatic confirmation of emergent misalignment.